# 01. Explore the ten tables

A teaching notebook: read it top to bottom to learn what is in the State Street case data, how the tables fit together, and where the first problems show up. Every number below is computed from the raw files in `data/`. Nothing in `data/` is changed; the only saved results go to `outputs/`.

**How to run it:** from the repo root, `uv run jupyter lab` and open this file, then *Run All*. It needs about 1.5 GB of memory and a minute or two.

**Confidential data:** the outputs of this notebook show real data rows. Do not commit an executed copy. The team keeps executed copies in `outputs/`, which Git ignores.

> ### What we found so far
>
> *Written from a full run of this notebook on 15 Oct 2026. Section 9.3 prints the same numbers live, so you can check them after any rerun.*
>
> - **The files are intact.** All ten files match the jury's `manifest.csv` (checksum, rows, columns, size): 10 of 10 PASS.
> - **`transaction_id` is not unique.** 1,409,533 rows but 1,407,428 distinct IDs: 2,105 IDs appear exactly twice, always in adjacent rows, and at the same row positions in `pipeline_events`. A naive join between the two tables creates 4,210 extra rows.
> - **Most repeats are harmless copies, a few are real conflicts.** Of the 2,105 pairs: 2,032 are exact copies, 28 differ only because one copy writes `trade_date` as DD/MM/YYYY, and **45 disagree on a fact** (settlement_date 21, price 14, quantity 6, currency 4, settlement_note 3). The conflicts are blanks, one sign flip, currency swaps, settlement dates that move, and prices off by exactly a factor of 100.
> - **Date formats are mixed.** 9,398 `trade_date` values are DD/MM/YYYY; 1,687 `settlement_date` values are blank. Nothing is unreadable once both formats are handled.
> - **The pipeline copies differ in timing.** 1,987 of the 2,105 pipeline pairs have a different `published_timestamp` (20 `booked_timestamp`, 7 `received_timestamp`); source, batch, event time and cutoff never differ.
> - **Our book has 480 holdings the custodian does not report.** Full outer merge: 480,183 keys in both, 480 only in `positions`, 0 only in `comparison_positions`. None of the 480 is zero.
> - **Positions point to master records that do not exist.** 195 account_ids (699 position rows) are missing from `accounts`; 15 security_ids (1,118 position rows) are missing from `securities`, and **all 15 are spin-off targets** in `corporate_actions`, which confirms our main spin-off finding from a second table.
> - **Two methods agree.** DuckDB reproduces the pandas counts, and all 51 scored numbers from our manual chat session match this notebook.

## Table of contents

0. Summary (above)
1. [Setup](#setup): find the repo, define the helpers
2. [Integrity check](#integrity): are the files exactly what the jury sent?
3. [Peek at each table](#peek): what one row means
4. [Keys and joins](#keys): is `transaction_id` really unique?
5. [The repeated transaction IDs](#repeats): copies, date formats and real conflicts
6. [Positions versus comparison_positions](#positions): our book against the custodian
7. [TODO sections](#todo): statuses, position rebuild, price/FX/ledger
8. [Defect log](#defects): a table to copy into the defect catalog
9. [Verification and cross-check](#verification): DuckDB second method and our manual results

<a id="setup"></a>
## 1. Setup

The notebook lives in `notebooks/`, but the data lives one folder up. The next cell imports the libraries and finds the repo root by walking up from the current folder until it finds a folder that contains both `manifest.csv` and `data/`. This way the notebook works whether Jupyter was started in the repo root or in `notebooks/`.

In [ ]:
from pathlib import Path
import csv
import gzip
import hashlib
import platform

import duckdb
import pandas as pd

pd.set_option('display.max_columns', 30)      # show all columns of our widest table
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', None)   # do not cut off long descriptions


def find_repo_root(start=None):
    """Walk up from `start` until we find the folder that holds manifest.csv and data/."""
    folder = Path(start or Path.cwd()).resolve()
    for candidate in [folder, *folder.parents]:   # the folder itself, then its parent, grandparent, ...
        if (candidate / 'manifest.csv').is_file() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError(f'No folder with manifest.csv and data/ above {folder}')


ROOT = find_repo_root()
OUTPUTS = ROOT / 'outputs'
print('Repo root found:', ROOT.name)
print('Python', platform.python_version(), '| pandas', pd.__version__, '| duckdb', duckdb.__version__)

Next we copy three small helpers from `load_data.ipynb` (the loader the jury gave us). A notebook cannot be imported like a Python module, so copying them is the simplest honest option; they are unchanged except that they use our `ROOT`.

- `manifest()` reads `manifest.csv` (the jury's list of files, row counts and checksums).
- `table_path(name)` turns a table name into the path of its `.csv.gz` file.
- `read_table(name)` loads a table **as text**: `dtype=str` keeps every value as a string, and `keep_default_na=False` stops pandas from turning blanks into `NaN`. This matters because IDs like `000123` would lose their leading zeros as numbers, and a blank is evidence we want to see, not hide.

In [ ]:
# Copied from load_data.ipynb (section 2), unchanged apart from using our ROOT.
def manifest(root=ROOT):
    with (Path(root) / 'manifest.csv').open(newline='', encoding='utf-8') as stream:
        return {row['table']: row for row in csv.DictReader(stream)}

def table_path(name, root=ROOT):
    records = manifest(root)
    if name not in records:
        raise ValueError(f'Unknown table {name!r}. Choose from {sorted(records)}')
    return Path(root) / records[name]['file']

def read_table(name, root=ROOT):
    """Return a pandas frame; preserve all values as raw strings, including blanks."""
    return pd.read_csv(table_path(name, root), dtype=str, keep_default_na=False)

The data dictionary (`data_dictionary.csv`) describes every column of every table. The helper below shows the part of it for one table, so that each time we look at a table we also read what its columns are supposed to mean.

In [ ]:
dictionary = pd.read_csv(ROOT / 'data_dictionary.csv', dtype=str, keep_default_na=False)

def describe_columns(table_name):
    """Return the data dictionary rows (column, intended type, description) for one table."""
    rows = dictionary[dictionary['table'] == table_name]
    return rows[['column', 'intended_type', 'description']].reset_index(drop=True)

describe_columns('fx_rates')   # a small example

**Why we need a date helper.** The Dataset Guide says dates come in two formats: `YYYY-MM-DD` (for example `2026-06-03`) and `DD/MM/YYYY` (for example `03/06/2026`). Both mean 3 June 2026, but as text they are different, so a simple `==` would call them different. We also must not let pandas *guess* the format: `03/06/2026` could be read as 6 March in a US locale.

`normalise_date` tries the ISO format first, then `DD/MM/YYYY` for whatever failed, and returns ISO text. A blank stays blank, and anything that fits neither format is kept **unchanged** (so nothing disappears and two odd values can still be compared). `normalise_timestamp` does the same for the date part of a `YYYY-MM-DD hh:mm:ss` timestamp. `date_format_counts` counts how many values are ISO, slash, blank or unreadable, so failures are always reported.

In [ ]:
def normalise_date(raw_dates):
    """Turn YYYY-MM-DD and DD/MM/YYYY text into YYYY-MM-DD text.
    Blanks stay blank; values in neither format are returned unchanged."""
    iso = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce')      # NaT where it is not ISO
    slash = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce')    # NaT where it is not DD/MM/YYYY
    parsed = iso.fillna(slash)                                               # take ISO first, slash second
    as_text = parsed.dt.strftime('%Y-%m-%d')                                 # back to text, NaN where unparsed
    return as_text.fillna(raw_dates)                                         # keep the raw value where unparsed


def normalise_timestamp(raw_timestamps):
    """Normalise the first 10 characters (the date part) and keep the time part as it is."""
    date_part = raw_timestamps.str.slice(0, 10)
    time_part = raw_timestamps.str.slice(10)
    return normalise_date(date_part) + time_part


def date_format_counts(raw_dates):
    """Count how many values are ISO, DD/MM/YYYY, blank, or neither."""
    is_iso = pd.to_datetime(raw_dates, format='%Y-%m-%d', errors='coerce').notna()
    is_slash = pd.to_datetime(raw_dates, format='%d/%m/%Y', errors='coerce').notna()
    is_blank = raw_dates == ''
    return {'ISO YYYY-MM-DD': int(is_iso.sum()),
            'slash DD/MM/YYYY': int(is_slash.sum()),
            'blank': int(is_blank.sum()),
            'unreadable': int((~is_iso & ~is_slash & ~is_blank).sum())}


demo = pd.Series(['2026-06-03', '03/06/2026', '', 'not a date'])
pd.DataFrame({'raw': demo, 'normalised': normalise_date(demo)})

**What this tells us.** Both spellings of 3 June 2026 now become the same text, `2026-06-03`. The blank stays blank and the nonsense value is kept as it was, so it cannot silently disappear. We will use this helper every time we compare dates.

<a id="integrity"></a>
## 2. Integrity check

Before analysing anything we check that the files on our disk are exactly the files the jury published. `verify()` is copied unchanged from `load_data.ipynb`. For each of the ten files it compares four things with `manifest.csv`: the SHA-256 checksum (a fingerprint of the file's bytes), the number of rows, the number of columns and the file size. It reads every file, so it takes a little while. A PASS proves the delivery is intact; it says nothing about whether the records inside are correct.

In [ ]:
# Copied from load_data.ipynb (section 2), unchanged.
def verify(root=ROOT):
    """Check transport integrity and physical counts, not data quality."""
    failures = []
    for name, expected in manifest(root).items():
        path = table_path(name, root)
        if not path.is_file():
            failures.append(f'{name}: file missing')
            continue
        sha = hashlib.sha256()
        with path.open('rb') as stream:
            for block in iter(lambda: stream.read(1024 * 1024), b''):
                sha.update(block)
        with gzip.open(path, 'rt', newline='', encoding='utf-8') as stream:
            reader = csv.reader(stream)
            header = next(reader)
            count = sum(1 for _ in reader)
        ok = (sha.hexdigest() == expected['sha256']
              and count == int(expected['rows'])
              and len(header) == int(expected['columns'])
              and path.stat().st_size == int(expected['bytes']))
        print(f'{name}: {count:,} rows; {"PASS" if ok else "FAIL"}', flush=True)
        if not ok:
            failures.append(name)
    if failures:
        raise SystemExit('Release verification failed: ' + ', '.join(failures))
    print('All ten files verified. This does not establish correctness of their records.')

verify()

**What this tells us.** All ten files PASS: they are byte for byte the files the jury published. Any problem we find later is in the data itself, not caused by a damaged download.

Now we load all ten tables into pandas, as text, and keep them in a dictionary called `tables` (and as separate variables with the table names, for convenience). Then we compare the shape of each loaded table (rows and columns) with `manifest.csv`. This is the official check: the manifest comes from the jury, so a FAIL here would mean we loaded something differently from them.

In [ ]:
TABLE_NAMES = ['accounts', 'securities', 'transactions', 'pipeline_events', 'positions',
               'comparison_positions', 'corporate_actions', 'prices', 'fx_rates', 'ledger_nav']

tables = {}
for name in TABLE_NAMES:
    tables[name] = read_table(name)

accounts = tables['accounts']
securities = tables['securities']
transactions = tables['transactions']
pipeline_events = tables['pipeline_events']
positions = tables['positions']
comparison_positions = tables['comparison_positions']
corporate_actions = tables['corporate_actions']
prices = tables['prices']
fx_rates = tables['fx_rates']
ledger_nav = tables['ledger_nav']

official = manifest()
shape_check = pd.DataFrame([
    {'table': name,
     'manifest_rows': int(official[name]['rows']),
     'loaded_rows': len(tables[name]),
     'manifest_columns': int(official[name]['columns']),
     'loaded_columns': tables[name].shape[1]}
    for name in TABLE_NAMES
])
rows_ok = shape_check['manifest_rows'] == shape_check['loaded_rows']
columns_ok = shape_check['manifest_columns'] == shape_check['loaded_columns']
shape_check['result'] = (rows_ok & columns_ok).map({True: 'PASS', False: 'FAIL'})
shape_check

**What this tells us.** Loading with pandas gives exactly the row and column counts in `manifest.csv` for all ten tables (10 of 10 PASS). Reading everything as text lost no rows, so we start from the complete release.

<a id="peek"></a>
## 3. Peek at each table

One table at a time: first what the data dictionary says about its columns, then the first five raw rows exactly as they are in the file. The aim is to know what a single row *means* (its "grain") before we join anything. We start with the master data (accounts, securities), then the trade flow (transactions, pipeline_events), then holdings (positions, comparison_positions), then reference and valuation data.

### 3.1 `accounts`

The list of client accounts: who holds what, in which fund and share class. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('accounts'))
print(f'accounts: {len(accounts):,} rows, {accounts.shape[1]} columns')
accounts.head(5)

**What this tells us.** **One row = one client account**, with its fund (`fund_id`), share class, base currency and custodian. A blank `close_date` means the account is still open, not that data is missing. `fund_id` is the link to the ledger later.

### 3.2 `securities`

The security master: every instrument the platform is supposed to know about. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('securities'))
print(f'securities: {len(securities):,} rows, {securities.shape[1]} columns')
securities.head(5)

**What this tells us.** **One row = one instrument** the platform knows. Equities have blank `coupon_rate`, `maturity_date` and `par_value` (they do not apply); bonds have them filled. `security_id` is the key we join on; CUSIP and ISIN are only descriptive attributes.

### 3.3 `transactions`

Every trade (BUY or SELL) booked during the observation window. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('transactions'))
print(f'transactions: {len(transactions):,} rows, {transactions.shape[1]} columns')
transactions.head(5)

**What this tells us.** **One row = one trade**: an account buys or sells a quantity of a security at a price, on a trade date, to settle on a settlement date. Already in the first five rows one `trade_date` is written as DD/MM/YYYY, which is why we built the date helper.

### 3.4 `pipeline_events`

The timeline of each trade through the platform: when it was received, booked and published. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('pipeline_events'))
print(f'pipeline_events: {len(pipeline_events):,} rows, {pipeline_events.shape[1]} columns')
pipeline_events.head(5)

**What this tells us.** **One row = the journey of one trade through the platform**: event, received, booked, published, and the cutoff it had to meet. It shares `transaction_id` with `transactions`, so it is the place to measure timeliness.

### 3.5 `positions`

Our own book of record: how many units each account holds of each security on the snapshot date. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('positions'))
print(f'positions: {len(positions):,} rows, {positions.shape[1]} columns')
positions.head(5)

**What this tells us.** **One row = how many units one account holds of one security on 10 July 2026**, according to our own book. Negative quantities appear right away; the Dataset Guide says these can be short positions, not automatically errors.

### 3.6 `comparison_positions`

The same holdings as reported by the custodian, an outside source we reconcile against. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('comparison_positions'))
print(f'comparison_positions: {len(comparison_positions):,} rows, {comparison_positions.shape[1]} columns')
comparison_positions.head(5)

**What this tells us.** **One row = the same kind of holding, as reported by the custodian.** It has 480 fewer rows than `positions`, so some holdings exist on only one side. Section 6 finds out which.

### 3.7 `corporate_actions`

Issuer events (splits, mergers, spin-offs, redemptions, dividends) that can change holdings. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('corporate_actions'))
print(f'corporate_actions: {len(corporate_actions):,} rows, {corporate_actions.shape[1]} columns')
corporate_actions.head(5)

**What this tells us.** **One row = one issuer event** (split, merger, spin-off, redemption, cash dividend). In the first rows only the spin-off has a `target_security_id`; the merger has none, which a later check should look at. Note that the table says nothing about which accounts received the event.

### 3.8 `prices`

Prices from several vendors per security and day. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('prices'))
print(f'prices: {len(prices):,} rows, {prices.shape[1]} columns')
prices.head(5)

**What this tells us.** **One row = one vendor's price for one security on one day.** In the first rows one security has three vendor prices per day, ranked by `vendor_priority` (1 = preferred, not proven correct).

### 3.9 `fx_rates`

Exchange rates to USD per currency and day. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('fx_rates'))
print(f'fx_rates: {len(fx_rates):,} rows, {fx_rates.shape[1]} columns')
fx_rates.head(5)

**What this tells us.** **One row = the USD value of one unit of a currency on one day.** Even the USD rate is not exactly 1 in these rows, the open question already noted in the project summary.

### 3.10 `ledger_nav`

The fund ledger: net assets, shares outstanding and NAV per share per fund and share class. The next cell shows the column descriptions, the number of rows and the first five raw rows.

In [ ]:
display(describe_columns('ledger_nav'))
print(f'ledger_nav: {len(ledger_nav):,} rows, {ledger_nav.shape[1]} columns')
ledger_nav.head(5)

**What this tells us.** **One row = one share class of one fund on 10 July 2026**, with net assets, shares outstanding and NAV per share. This is the final output the whole platform feeds.

<a id="keys"></a>
## 4. Keys and joins

A **key** is a column (or set of columns) that should identify exactly one row. The Dataset Guide says `transaction_id` is the *intended* key of both `transactions` and `pipeline_events`, and that the link between them is "intended one to one; validate first". So we validate it.

The next cell counts, for both tables: the number of rows, the number of distinct IDs, `duplicated().sum()` (how many rows are a second or later copy of an ID already seen), and how many different IDs repeat. If the key were unique, rows would equal distinct IDs and the last two numbers would be zero.

In [ ]:
def id_summary(table, column):
    """Count rows, distinct values and repeats of one ID column."""
    ids = table[column]
    return {'rows': len(ids),
            'distinct IDs': ids.nunique(),
            'extra copies (duplicated().sum())': int(ids.duplicated().sum()),
            'IDs that repeat': ids[ids.duplicated()].nunique()}

key_check = pd.DataFrame({'transactions': id_summary(transactions, 'transaction_id'),
                          'pipeline_events': id_summary(pipeline_events, 'transaction_id')})
key_check

**What this tells us.** Both tables have 1,409,533 rows but only 1,407,428 distinct IDs: 2,105 IDs repeat, each adding exactly one extra row. So `transaction_id` is **not** a unique key, and the two tables have the same number of repeats.

Next we check coverage in **both directions** with `isin`: does every transaction ID appear in `pipeline_events`, and does every pipeline ID appear in `transactions`? An ID found on only one side would be an orphan: a trade with no pipeline history, or a pipeline record for a trade that does not exist.

In [ ]:
txn_found_in_pipeline = transactions['transaction_id'].isin(pipeline_events['transaction_id'])
pipeline_found_in_txn = pipeline_events['transaction_id'].isin(transactions['transaction_id'])

print('transactions rows whose ID is missing from pipeline_events:', (~txn_found_in_pipeline).sum())
print('pipeline_events rows whose ID is missing from transactions:', (~pipeline_found_in_txn).sum())

same_id_sets = set(transactions['transaction_id']) == set(pipeline_events['transaction_id'])
print('Both tables contain exactly the same set of IDs:', same_id_sets)

**What this tells us.** No orphans in either direction: every trade has pipeline history and every pipeline record belongs to a trade. The only problem with the link is the repeats.

**Why repeated keys are dangerous in a join: many-to-many expansion.** When you merge two tables on a key, pandas pairs *every* matching row on the left with *every* matching row on the right. If an ID appears twice on each side, it produces 2 x 2 = 4 rows, not 2. The toy example below uses made-up IDs (`toy_A`, `toy_B`, not from our data): 3 rows on each side become 5 rows after the merge.

In [ ]:
left = pd.DataFrame({'transaction_id': ['toy_A', 'toy_B', 'toy_B'], 'quantity': [100, 50, 50]})
right = pd.DataFrame({'transaction_id': ['toy_A', 'toy_B', 'toy_B'], 'batch_id': ['B1', 'B2', 'B2']})

toy_joined = left.merge(right, on='transaction_id', how='outer')
print('rows before:', len(left), 'and', len(right), '| rows after:', len(toy_joined))
toy_joined

**What this tells us.** `toy_A` (once on each side) gives 1 row; `toy_B` (twice on each side) gives 2 x 2 = 4 rows. That is how a join can quietly create rows that do not exist, and double count quantities in a position rebuild.

Now the real join. We merge only the ID column of each table (to save memory) with `how='outer'`, so nothing is dropped. Before running it we *predict* the row count: for each ID, (copies in transactions) x (copies in pipeline_events), summed over all IDs. If the prediction matches the actual result, we fully understand where the extra rows come from. The Dataset Guide asks us to report row counts before and after every join; this is why.

In [ ]:
copies_in_txn = transactions['transaction_id'].value_counts()        # copies per ID in transactions
copies_in_pipe = pipeline_events['transaction_id'].value_counts()    # copies per ID in pipeline_events
predicted_rows = (copies_in_txn * copies_in_pipe.reindex(copies_in_txn.index, fill_value=0)).sum()

real_joined = transactions[['transaction_id']].merge(
    pipeline_events[['transaction_id']], on='transaction_id', how='outer', indicator=True)

print(f'rows before: transactions {len(transactions):,}, pipeline_events {len(pipeline_events):,}')
print(f'rows after the outer merge: {len(real_joined):,}')
print(f'predicted from copies per ID: {predicted_rows:,}')
print(f'extra rows created by the join: {len(real_joined) - len(transactions):,}')
print(real_joined['_merge'].value_counts().to_string())
del real_joined   # free memory, we only needed the counts

**What this tells us.** The real merge grows from 1,409,533 to 1,413,743 rows (+4,210), exactly as predicted: each of the 2,105 repeated IDs turns 2 rows into 4. No row is `left_only` or `right_only`. Any later join on `transaction_id` must handle the repeats first.

<a id="repeats"></a>
## 5. The repeated transaction IDs

Now we look closely at the transaction IDs that appear more than once. The questions are: how many copies does each have, where are the copies in the file, are the copies identical, and if not, *how* do they differ?

First we keep only the rows whose ID repeats (`duplicated(keep=False)` marks *every* copy, not only the second one) and remember each row's position in the raw file. The row position is lineage only (where the record sat in the file), never a business key.

In [ ]:
repeated_rows = transactions[transactions['transaction_id'].duplicated(keep=False)].copy()
repeated_rows['row_number'] = repeated_rows.index   # position in the raw file (0 = first data row)

copies_per_id = repeated_rows.groupby('transaction_id').size()
print(f'rows with a repeated ID: {len(repeated_rows):,}')
print(f'distinct repeated IDs: {repeated_rows["transaction_id"].nunique():,}')
print('How many copies each repeated ID has (copies -> number of IDs):')
print(copies_per_id.value_counts().to_string())

**What this tells us.** 4,210 rows carry a repeated ID: 2,105 IDs, and every one appears **exactly twice**. There are no triples, which makes the pairwise comparison below complete.

Where do the copies sit in the file? For each repeated ID we compute the gap between the row numbers of its first and last copy. A gap of 1 means the copies are in adjacent rows. Then we check whether `pipeline_events` has its repeated IDs at exactly the same row positions, which would suggest both files were produced by the same step (for example one record sent twice).

In [ ]:
row_gap = repeated_rows.groupby('transaction_id')['row_number'].agg(lambda rows: rows.max() - rows.min())
print('Gap between first and last copy (gap -> number of IDs):')
print(row_gap.value_counts().to_string())

repeated_pipe = pipeline_events[pipeline_events['transaction_id'].duplicated(keep=False)].copy()
repeated_pipe['row_number'] = repeated_pipe.index

same_positions = list(repeated_rows['row_number']) == list(repeated_pipe['row_number'])
same_ids_at_positions = list(repeated_rows['transaction_id']) == list(repeated_pipe['transaction_id'])
print('Repeated rows sit at the same row positions in both tables:', same_positions)
print('...and carry the same IDs at those positions:', same_ids_at_positions)

**What this tells us.** Every pair sits in **adjacent rows** (gap of 1), and `pipeline_events` has its repeats at the very same row positions with the same IDs. This looks like one record emitted twice by the same upstream step and copied into both files, rather than two independent trades that happen to share an ID.

**Exact copies versus differing copies.** A row is an *exact copy* if another row with the same ID has the same value in every column. `duplicated(keep=False)` on all columns marks every row that has an identical twin.

In [ ]:
data_columns = list(transactions.columns)   # the 16 columns of the file (not our row_number)
is_exact_copy = repeated_rows.duplicated(subset=data_columns, keep=False)

exact_copy_rows = int(is_exact_copy.sum())
exact_copy_pairs = repeated_rows.loc[is_exact_copy, 'transaction_id'].nunique()
print(f'rows that are exact full-row copies: {exact_copy_rows:,} ({exact_copy_pairs:,} IDs)')
print(f'rows whose copies differ somewhere: {len(repeated_rows) - exact_copy_rows:,}')

**What this tells us.** 4,064 rows (2,032 pairs) are perfect copies: identical in all 16 columns. The other 146 rows (73 pairs) differ somewhere, and those are the ones worth a closer look.

To compare copies column by column we put them side by side. `groupby(...).cumcount()` numbers the copies of each ID in file order (0 for the first, 1 for the second). Then `copy_a` holds every first copy and `copy_b` every second copy, both indexed by `transaction_id` so they line up. The code also reports the largest number of copies; if any ID had more than two, copies after the second would not be compared here and we would say so.

In [ ]:
repeated_rows['copy_number'] = repeated_rows.groupby('transaction_id').cumcount()   # 0, 1, ... within each ID

largest_group = copies_per_id.max()
print('largest number of copies of one ID:', largest_group)
if largest_group > 2:
    print('Note: some IDs have more than two copies; only the first two are compared below.')

copy_a = repeated_rows[repeated_rows['copy_number'] == 0].set_index('transaction_id')
copy_b = repeated_rows[repeated_rows['copy_number'] == 1].set_index('transaction_id')
copy_b = copy_b.loc[copy_a.index]   # same order as copy_a so each row is the same ID

compare_columns = [column for column in data_columns if column != 'transaction_id']
raw_differs = pd.DataFrame({column: copy_a[column] != copy_b[column] for column in compare_columns})
print(f'pairs compared: {len(copy_a):,}')
print('Pairs whose raw text differs, per column:')
raw_differs.sum()[raw_differs.sum() > 0]

**What this tells us.** As raw text, the 73 differing pairs disagree in `trade_date` (30), `settlement_date` (21), `price` (14), `quantity` (6), `currency` (4) and `settlement_note` (3). We do not yet know which of these are only a different way of writing the same date.

Some of these differences may be the *same date written two ways*. Before normalising, it helps to see how common the two formats are in the **whole** `transactions` table, for each date column (and the date part of `event_timestamp`).

In [ ]:
format_table = pd.DataFrame({
    'trade_date': date_format_counts(transactions['trade_date']),
    'settlement_date': date_format_counts(transactions['settlement_date']),
    'event_timestamp (date part)': date_format_counts(transactions['event_timestamp'].str.slice(0, 10)),
})
format_table

**What this tells us.** In the whole table, 9,398 `trade_date` values use DD/MM/YYYY; `settlement_date` and `event_timestamp` are always ISO. `settlement_date` is blank 1,687 times. No value is unreadable, so our helper handles every date in this table.

Now we normalise the date columns of both copies with our helper and compare again. A pair that differed in the raw text but is identical after normalising differs **only by date format**: the same record written twice in two formats. Whatever still differs is a **real conflict**: the two copies disagree about a fact. A pair can conflict in more than one column, so the per-column counts can add up to more than the number of pairs.

In [ ]:
def normalise_transaction_dates(frame):
    """Return a copy of a transactions frame with its date columns in YYYY-MM-DD form."""
    frame = frame.copy()
    frame['trade_date'] = normalise_date(frame['trade_date'])
    frame['settlement_date'] = normalise_date(frame['settlement_date'])
    frame['event_timestamp'] = normalise_timestamp(frame['event_timestamp'])
    return frame

norm_a = normalise_transaction_dates(copy_a)
norm_b = normalise_transaction_dates(copy_b)
real_differs = pd.DataFrame({column: norm_a[column] != norm_b[column] for column in compare_columns})

pair_has_raw_difference = raw_differs.any(axis=1)
pair_has_real_conflict = real_differs.any(axis=1)
pair_differs_only_by_format = pair_has_raw_difference & ~pair_has_real_conflict

pair_types = pd.Series({
    'exact copies': int((~pair_has_raw_difference).sum()),
    'differ only by date format': int(pair_differs_only_by_format.sum()),
    'real conflict (at least one column)': int(pair_has_real_conflict.sum()),
})
print(pair_types.to_string())
print('total pairs:', pair_types.sum())
print()
print('Real conflicts per column after normalising (pairs can overlap):')
conflicts_per_column = real_differs.sum()
conflicts_per_column

**What this tells us.** After normalising, all 30 `trade_date` differences disappear. The 2,105 pairs split into **2,032 exact copies, 28 that differ only by date format, and 45 real conflicts**. The real conflicts are in `settlement_date` (21), `price` (14), `quantity` (6), `currency` (4) and `settlement_note` (3); `trade_date` and `event_timestamp` never conflict. The per-column counts add to 48, more than 45, so a few pairs conflict in more than one column.

**What kind of conflict is it?** For each conflicting pair and column we put a label on the difference, using simple rules written before looking at the examples:

- **blank vs value**: one copy is empty, the other is not;
- **currency difference**: the `currency` column has two different codes;
- **sign flip**: both are numbers and one is exactly minus the other (for example 4308 and -4308);
- **other change**: anything else (for example two different settlement dates).

In [ ]:
def as_number(text):
    """Convert text to a float, or None if it is not a number."""
    try:
        return float(text)
    except ValueError:
        return None


def conflict_kind(column, value_a, value_b):
    if value_a == '' or value_b == '':
        return 'blank vs value'
    if column == 'currency':
        return 'currency difference'
    number_a, number_b = as_number(value_a), as_number(value_b)
    if number_a is not None and number_b is not None and number_a == -number_b:
        return 'sign flip'
    return 'other change'


conflict_records = []
for column in compare_columns:
    for txn_id in real_differs.index[real_differs[column]]:    # IDs whose copies conflict in this column
        value_a = norm_a.loc[txn_id, column]
        value_b = norm_b.loc[txn_id, column]
        conflict_records.append({'transaction_id': txn_id, 'column': column,
                                 'copy_a': value_a, 'copy_b': value_b,
                                 'kind': conflict_kind(column, value_a, value_b)})
conflicts = pd.DataFrame(conflict_records)

pd.crosstab(conflicts['column'], conflicts['kind'], margins=True, margins_name='total')

**What this tells us.** Of the 48 column conflicts, 25 are a blank in one copy (price 10, settlement_date 7, quantity 5, settlement_note 3), 4 are currency differences, 1 is a sign flip on quantity, and 18 are other changes (14 different settlement dates, 4 different prices). Blanks suggest one copy lost a field; the others mean the copies disagree about the trade itself.

One example per kind of conflict, picked by code: for each kind we take the first transaction ID in sorted order. For each example we print both raw copies (all columns), so you can see the conflict in context.

In [ ]:
examples = conflicts.sort_values(['kind', 'transaction_id']).groupby('kind').head(1)

for _, example in examples.iterrows():
    print(f"Kind: {example['kind']} | column: {example['column']} | "
          f"copy_a: {example['copy_a']!r} | copy_b: {example['copy_b']!r}")
    both_copies = repeated_rows[repeated_rows['transaction_id'] == example['transaction_id']]
    display(both_copies[['row_number'] + data_columns])

**What this tells us.** The examples show each kind in context. The rows are identical except for one field: a missing price, a currency changed (with price and gross amount unchanged, so at most one currency can be right), a settlement date moved to a Saturday before the trade date, and a quantity whose sign flipped on a BUY. In a position rebuild, keeping the wrong copy changes the result, so these pairs cannot be fixed by simply dropping duplicates.

A closer look at the price conflicts labelled "other change" (both copies have a price, but different ones). Dividing one price by the other shows whether the difference is random or a clean multiple, which would point to a scale (unit) problem such as a price quoted in cents instead of units.

In [ ]:
price_changes = conflicts[(conflicts['column'] == 'price') & (conflicts['kind'] == 'other change')].copy()
price_changes['ratio copy_b / copy_a'] = (price_changes['copy_b'].astype(float)
                                          / price_changes['copy_a'].astype(float)).round(4)
price_changes[['column', 'copy_a', 'copy_b', 'ratio copy_b / copy_a']]

**What this tells us.** The four non-blank price conflicts are not random: one copy is exactly **100 times** the other (three times up, once down). That is the signature of a scale error, such as a price in cents versus units. Which copy is right is unresolved, but `gross_amount` (quantity x price) can tell us in the next notebook.

Are the repeated rows special in their status? We compare the status mix of all transactions with the status mix of the repeated rows.

In [ ]:
status_overall = transactions['status'].value_counts()
status_repeated = repeated_rows['status'].value_counts()
pd.DataFrame({'all transactions': status_overall,
              'repeated rows': status_repeated}).fillna(0).astype(int)

**What this tells us.** The repeated rows are 4,206 BOOKED and 4 CANCELLED, and none REVERSED. Almost all of them are BOOKED, so they would enter a BOOKED-only position rebuild twice unless we deduplicate first.

**The copies in `pipeline_events`.** We pair the repeated pipeline rows the same way (first copy against second copy) and count, per column, how many pairs differ. We normalise the date part of every timestamp first, so a format difference is not counted as a real one.

In [ ]:
pipe_columns = [column for column in pipeline_events.columns if column != 'transaction_id']
timestamp_columns = [column for column in pipe_columns if column.endswith('_timestamp')]

repeated_pipe['copy_number'] = repeated_pipe.groupby('transaction_id').cumcount()
pipe_a = repeated_pipe[repeated_pipe['copy_number'] == 0].set_index('transaction_id')
pipe_b = repeated_pipe[repeated_pipe['copy_number'] == 1].set_index('transaction_id')
pipe_b = pipe_b.loc[pipe_a.index]

for column in timestamp_columns:
    pipe_a[column] = normalise_timestamp(pipe_a[column])
    pipe_b[column] = normalise_timestamp(pipe_b[column])

pipe_differs = pd.DataFrame({column: pipe_a[column] != pipe_b[column] for column in pipe_columns})
print(f'pipeline pairs compared: {len(pipe_a):,}')
print(f'pipeline pairs that are identical in every column: {(~pipe_differs.any(axis=1)).sum():,}')
pipe_differs_per_column = pipe_differs.sum()
pipe_differs_per_column

**What this tells us.** Only 118 pipeline pairs are fully identical. The copies differ in `published_timestamp` (1,987 pairs), `booked_timestamp` (20) and `received_timestamp` (7), and never in `source_system`, `batch_id`, `event_timestamp` or `cutoff_timestamp`. So the duplicate came from the same source event and batch, and the copies were then processed (mostly published) at different times.

How large are the `published_timestamp` differences? We convert both copies to real timestamps and look at the gap (second copy minus first copy). Values that cannot be read become missing and are counted, not dropped.

In [ ]:
published_a = pd.to_datetime(pipe_a['published_timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
published_b = pd.to_datetime(pipe_b['published_timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
print('unreadable published timestamps:', published_a.isna().sum() + published_b.isna().sum())

gap = (published_b - published_a)[pipe_differs['published_timestamp']]   # only pairs that differ
print(f'pairs with a different published_timestamp: {len(gap):,}')
gap.describe()

**What this tells us.** The publication gaps are mostly small: the median is minus 1 minute (the second copy was published one minute earlier) and half the gaps lie between minus 5 and plus 6 minutes. The extremes are large: about 15 hours earlier to 16 hours later. Which copy counts as "the" publication time matters for any timeliness KPI.

<a id="positions"></a>
## 6. Positions versus comparison_positions

`positions` is our book of record; `comparison_positions` is what the custodian reports. Reconciliation means comparing them key by key. The Dataset Guide's key is account + security + snapshot date, and it asks for a **full outer** comparison so that a holding present on only one side stays visible (and is never silently treated as zero).

First, the basics of each table: which snapshot dates appear, how many distinct accounts and securities, and whether the key is unique (the "grain check" the guide asks for before comparing).

In [ ]:
position_key = ['account_id', 'security_id', 'as_of_date']

basics = {}
for name, table in [('positions', positions), ('comparison_positions', comparison_positions)]:
    basics[name] = {
        'rows': len(table),
        'as_of_date values': ', '.join(sorted(table['as_of_date'].unique())),
        'distinct account_ids': table['account_id'].nunique(),
        'distinct security_ids': table['security_id'].nunique(),
        'rows repeating a key': int(table.duplicated(subset=position_key).sum()),
    }
pd.DataFrame(basics)

**What this tells us.** Both tables are one snapshot (`2026-07-10`) and cover the same 100,195 accounts and 6,015 securities. Neither has a repeated key, so the grain check passes and a full outer merge cannot create extra rows.

Now the full outer merge with `indicator=True`. The indicator adds a column `_merge` that says, for each key, whether it was found in both tables (`both`), only in positions (`left_only`) or only in the custodian file (`right_only`). We keep only the key and the quantity from each side and rename the quantities so we can tell them apart.

In [ ]:
book = positions[position_key + ['quantity']].rename(columns={'quantity': 'book_quantity'})
custodian = comparison_positions[position_key + ['quantity']].rename(columns={'quantity': 'custodian_quantity'})

position_compare = book.merge(custodian, on=position_key, how='outer', indicator=True)
side_names = {'both': 'in both', 'left_only': 'only in positions', 'right_only': 'only in comparison_positions'}
position_compare['side'] = position_compare['_merge'].map(side_names)

print(f'rows before: positions {len(book):,}, comparison_positions {len(custodian):,}')
print(f'rows after the outer merge: {len(position_compare):,}')
merge_counts = position_compare['side'].value_counts().reindex(list(side_names.values()), fill_value=0)
merge_counts

**What this tells us.** The merge has 480,663 rows, the same as `positions`, so no row was created or lost. 480,183 keys are in both tables, **480 are only in our book**, and none are only at the custodian. Every custodian holding is matched; the gap is entirely on our side.

The rows found **only in positions** are holdings we think we have but the custodian does not report. We convert their quantity to numbers (counting any value that fails) and summarise: how many, smallest, largest, median, mean, and how many are exactly zero. A zero would be a harmless "empty" position; a non-zero one is a real break to investigate.

In [ ]:
book_only = position_compare[position_compare['side'] == 'only in positions'].copy()
book_only['quantity_number'] = pd.to_numeric(book_only['book_quantity'], errors='coerce')

print('quantities that could not be read as numbers:', book_only['quantity_number'].isna().sum())
print('book-only rows with quantity exactly zero:', (book_only['quantity_number'] == 0).sum())
book_only_summary = book_only['quantity_number'].describe()
book_only_summary

**What this tells us.** All 480 quantities are readable numbers, and **none is zero**, so these are not empty positions that the custodian simply leaves out. They range from -12,322 to 17,690 (median 1,231.5, mean about 870.26). These are real breaks to trace in a later notebook.

**Orphan IDs.** Every `account_id` in positions should exist in `accounts`, and every `security_id` should exist in `securities`. First we confirm that the two master tables have unique keys (otherwise "exists" would be ambiguous), then we list the IDs in positions that have no master record, and count how many position rows they touch.

In [ ]:
print('repeated account_ids in accounts:', accounts['account_id'].duplicated().sum())
print('repeated security_ids in securities:', securities['security_id'].duplicated().sum())

position_account_ids = pd.Series(positions['account_id'].unique())
position_security_ids = pd.Series(positions['security_id'].unique())

orphan_account_ids = position_account_ids[~position_account_ids.isin(accounts['account_id'])]
orphan_security_ids = position_security_ids[~position_security_ids.isin(securities['security_id'])]

rows_with_orphan_account = int(positions['account_id'].isin(orphan_account_ids).sum())
rows_with_orphan_security = int(positions['security_id'].isin(orphan_security_ids).sum())

pd.DataFrame({
    'distinct IDs in positions': [len(position_account_ids), len(position_security_ids)],
    'missing from master table': [len(orphan_account_ids), len(orphan_security_ids)],
    'position rows affected': [rows_with_orphan_account, rows_with_orphan_security],
}, index=['account_id (vs accounts)', 'security_id (vs securities)'])

**What this tells us.** The master tables themselves have unique keys. But `positions` refers to **195 account_ids** (699 position rows) that do not exist in `accounts`, and **15 security_ids** (1,118 position rows) that do not exist in `securities`. That is why positions shows 100,195 accounts and 6,015 securities against 100,000 and 6,012 in the master tables.

Where could securities come from that are held but not in the security master? One candidate is a corporate action that *creates* a new security (a spin-off or merger target). The next cell checks how many of the orphan `security_id`s appear as a `target_security_id` in `corporate_actions`, and for which kind of action. This uses a different table as evidence, which is what the Working Guide asks for before explaining a flag.

In [ ]:
orphan_is_target = orphan_security_ids.isin(corporate_actions['target_security_id'])
print(f'orphan security_ids that are a corporate action target: {orphan_is_target.sum()} of {len(orphan_security_ids)}')

target_actions = corporate_actions[corporate_actions['target_security_id'].isin(orphan_security_ids)]
target_actions['action_type'].value_counts()

**What this tells us.** All 15 missing securities are `target_security_id`s of SPINOFF events. This is independent evidence from a second table for our main finding: spin-offs created new securities that were never set up in the security master, yet are held in 1,118 positions.

<a id="todo"></a>
## 7. TODO sections (not implemented yet)

These are placeholders for the next notebooks. Nothing below is computed yet.

### 7.1 TODO: transaction statuses and `related_transaction_id`
- Count BOOKED, CANCELLED and REVERSED, and how often `related_transaction_id` is filled for each.
- Check that every `related_transaction_id` exists in `transactions`, and what status the linked record has.
- A CANCELLED original linked to a BOOKED replacement can be a valid rebook (Dataset Guide, section 3).
- REVERSED can mean "inactive" or "offsetting trade": show both readings and avoid double offsets.

### 7.2 TODO: dry-run position rebuild for one account
- Pick one account by code (for example the first account in sorted order that has both BUY and SELL trades).
- Start from zero before 1 June (the Dataset Guide's shared benchmark, not a claim about real portfolios).
- BUY adds the quantity, SELL subtracts it; BOOKED trades only as the first, transparent version.
- Do not take absolute values of negative raw quantities; investigate them.
- Compare the result with `positions` for that account and explain every difference.

### 7.3 TODO: price, FX and ledger checks
- Prices: coverage per security and date, freshness, spread between vendors, scale (unit) differences.
- FX: one rate per currency and date; `rate_to_usd` is USD per one unit of currency.
- Ledger: test `net_assets / shares_outstanding = nav_per_share`; never sum `nav_per_share`.

<a id="defects"></a>
## 8. Defect log

A small table of what this notebook found, built from the variables computed above (no numbers typed by hand). The columns follow the Working Guide: what the problem is, where, how many rows, how we detected it, and the evidence status (**confirmed**, **unresolved** or **cleared**). It is saved to `outputs/01_defect_log.csv` so it can be copied into the defect catalog.

In [ ]:
format_only_rows = int(pair_differs_only_by_format.sum()) * 2   # two rows per pair
real_conflict_rows = int(pair_has_real_conflict.sum()) * 2
pipe_repeated_rows = len(repeated_pipe)

defect_log = pd.DataFrame([
    {'defect': 'Repeated transaction_id, exact full-row copy',
     'table': 'transactions', 'rows': exact_copy_rows,
     'how detected': 'transaction_id appears more than once and all 16 columns are equal',
     'evidence status': 'confirmed'},
    {'defect': 'Repeated transaction_id, copies differ only by date format',
     'table': 'transactions', 'rows': format_only_rows,
     'how detected': 'copies differ as text but are equal after normalising YYYY-MM-DD and DD/MM/YYYY',
     'evidence status': 'confirmed'},
    {'defect': 'Repeated transaction_id, copies conflict on a fact',
     'table': 'transactions', 'rows': real_conflict_rows,
     'how detected': 'copies still differ after date normalisation (blank, sign flip, currency, other)',
     'evidence status': 'unresolved (which copy is right is unknown)'},
    {'defect': 'Repeated transaction_id in the pipeline',
     'table': 'pipeline_events', 'rows': pipe_repeated_rows,
     'how detected': 'transaction_id appears more than once; same row positions as in transactions',
     'evidence status': 'confirmed'},
    {'defect': 'trade_date written as DD/MM/YYYY instead of YYYY-MM-DD',
     'table': 'transactions', 'rows': format_table.loc['slash DD/MM/YYYY', 'trade_date'],
     'how detected': 'date format count on the full table',
     'evidence status': 'confirmed (format inconsistency, readable)'},
    {'defect': 'Blank settlement_date',
     'table': 'transactions', 'rows': format_table.loc['blank', 'settlement_date'],
     'how detected': 'date format count on the full table',
     'evidence status': 'confirmed (missing value)'},
    {'defect': 'Position in our book with no custodian record',
     'table': 'positions', 'rows': int(merge_counts['only in positions']),
     'how detected': 'full outer merge on account_id, security_id, as_of_date',
     'evidence status': 'unresolved (cause not yet traced)'},
    {'defect': 'Position on an account_id missing from accounts',
     'table': 'positions', 'rows': rows_with_orphan_account,
     'how detected': 'positions.account_id not found in accounts.account_id',
     'evidence status': 'confirmed'},
    {'defect': 'Position on a security_id missing from securities',
     'table': 'positions', 'rows': rows_with_orphan_security,
     'how detected': ('positions.security_id not found in securities.security_id; '
                      f'{orphan_is_target.sum()} of {len(orphan_security_ids)} are corporate action targets'),
     'evidence status': 'confirmed'},
])

OUTPUTS.mkdir(exist_ok=True)
defect_log.to_csv(OUTPUTS / '01_defect_log.csv', index=False)
defect_log

**What this tells us.** Nine defect lines, all computed above. Seven are confirmed (the fact is directly observed); two are unresolved: for the 45 conflicting pairs we do not know which copy is right, and for the 480 book-only positions we have not traced the cause yet. The file is in `outputs/01_defect_log.csv`.

<a id="verification"></a>
## 9. Verification and cross-check

### 9.1 Second method: DuckDB

To make sure the pandas numbers are not an artefact of how we wrote the code, we compute three of them a second, independent way with DuckDB (SQL run directly on the `.csv.gz` files). As the repo rules require, DuckDB reads every column as text (`all_varchar=true`). If both methods agree, we can trust the number more.

In [ ]:
con = duckdb.connect()   # an in-memory database: nothing is written to disk

def duck_source(name):
    """SQL snippet that reads one table as text."""
    return f"read_csv('{table_path(name).as_posix()}', all_varchar=true, header=true)"

duck_distinct_ids = con.sql(f"""
    SELECT COUNT(DISTINCT transaction_id) FROM {duck_source('transactions')}
""").fetchone()[0]

duck_repeated_ids = con.sql(f"""
    SELECT COUNT(*) FROM (
        SELECT transaction_id FROM {duck_source('transactions')}
        GROUP BY transaction_id HAVING COUNT(*) > 1
    )
""").fetchone()[0]

duck_merge = con.sql(f"""
    SELECT CASE WHEN c.account_id IS NULL THEN 'only in positions'
                WHEN p.account_id IS NULL THEN 'only in comparison_positions'
                ELSE 'in both' END AS side,
           COUNT(*) AS n
    FROM {duck_source('positions')} AS p
    FULL OUTER JOIN {duck_source('comparison_positions')} AS c
      ON p.account_id = c.account_id
     AND p.security_id = c.security_id
     AND p.as_of_date = c.as_of_date
    GROUP BY side
""").df().set_index('side')['n']

pandas_vs_duckdb = pd.DataFrame([
    {'check': 'distinct transaction_ids', 'pandas': key_check.loc['distinct IDs', 'transactions'],
     'duckdb': duck_distinct_ids},
    {'check': 'repeated transaction_ids', 'pandas': key_check.loc['IDs that repeat', 'transactions'],
     'duckdb': duck_repeated_ids},
] + [
    {'check': f'positions merge: {side}', 'pandas': merge_counts[side],
     'duckdb': int(duck_merge.get(side, 0))}
    for side in side_names.values()
])
pandas_vs_duckdb['agree?'] = (pandas_vs_duckdb['pandas'] == pandas_vs_duckdb['duckdb']).map({True: 'yes', False: 'NO'})
pandas_vs_duckdb

**What this tells us.** DuckDB, reading the raw files with SQL, gives exactly the same five numbers as pandas: 1,407,428 distinct IDs, 2,105 repeated IDs, and 480,183 / 480 / 0 for the positions merge. The results do not depend on how we wrote the pandas code.

### 9.2 Cross-check against our manual results

Before this notebook, we worked some numbers out by hand in a chat session. They are typed into the dictionary below **only** so we can compare; nothing above uses them. The table shows each check, the manual result, this notebook's result and whether they match. Where a number was only an estimate, the last column says so. A mismatch is not something to hide or fix: it is shown, and explained in the cell after the table. It is saved to `outputs/01_cross_check.csv`.

In [ ]:
# Results from our earlier chat session: a second opinion only, NOT used anywhere above.
MANUAL_RESULTS = {
    'rows: transactions': 1_409_533,
    'rows: pipeline_events': 1_409_533,
    'rows: positions': 480_663,
    'rows: comparison_positions': 480_183,
    'rows: accounts': 100_000,
    'rows: securities': 6_012,
    'rows: corporate_actions': 200,
    'rows: prices': 539_970,
    'rows: fx_rates': 180,
    'rows: ledger_nav': 2_000,
    'status BOOKED': 1_403_784,
    'status CANCELLED': 4_347,
    'status REVERSED': 1_402,
    'distinct transaction_ids': 1_407_428,
    'repeated transaction_ids': 2_105,
    'each repeated ID appears exactly twice': 'yes',
    'copies sit in adjacent rows': 'yes',
    'same row positions in both tables': 'yes',
    'every ID in one table exists in the other': 'yes',
    'rows that are exact full-row copies': 4_064,
    'trade_date conflicts after normalising': 0,
    'event_timestamp conflicts after normalising': 0,
    'pairs with a real conflict': 45,
    'conflicts: settlement_date': 21,
    'conflicts: price': 14,
    'conflicts: quantity': 6,
    'conflicts: currency': 4,
    'conflicts: settlement_note': 3,
    'repeated rows: BOOKED': 4_206,
    'repeated rows: CANCELLED': 4,
    'pipeline pairs differ: published_timestamp': 1_987,
    'pipeline pairs differ: booked_timestamp': 20,
    'pipeline pairs differ: received_timestamp': 7,
    'pipeline pairs differ: source_system': 0,
    'pipeline pairs differ: batch_id': 0,
    'pipeline pairs differ: event_timestamp': 0,
    'pipeline pairs differ: cutoff_timestamp': 0,
    'positions as_of_date values': '2026-07-10',
    'comparison_positions as_of_date values': '2026-07-10',
    'positions distinct account_ids': 100_195,
    'comparison_positions distinct account_ids': 100_195,
    'positions distinct security_ids': 6_015,
    'comparison_positions distinct security_ids': 6_015,
    'merge: in both': 480_183,
    'merge: only in positions': 480,
    'merge: only in comparison_positions': 0,
    'book-only rows with zero quantity': 0,
    'book-only quantity min': -12_322,
    'book-only quantity max': 17_690,
    'book-only quantity median (about)': 1_231.5,
    'book-only quantity mean (about)': 870.26,
}
# Numbers we only estimated: shown for interest, not scored as match or mismatch.
MANUAL_ESTIMATES = {
    'pairs differing only by date format': 'about 28',
    'account_ids in positions missing from accounts': 'at least 195',
    'security_ids in positions missing from securities': 'at least 3',
}

The next cell collects this notebook's own result for each of those checks, from the variables computed above, and builds the comparison table. Means and medians are rounded to 2 decimals before comparing, because the manual values were given as "about".

In [ ]:
def yes_no(condition):
    return 'yes' if condition else 'no'

NOTEBOOK_RESULTS = {f'rows: {name}': len(tables[name]) for name in TABLE_NAMES}
NOTEBOOK_RESULTS.update({
    'status BOOKED': int(status_overall.get('BOOKED', 0)),
    'status CANCELLED': int(status_overall.get('CANCELLED', 0)),
    'status REVERSED': int(status_overall.get('REVERSED', 0)),
    'distinct transaction_ids': int(key_check.loc['distinct IDs', 'transactions']),
    'repeated transaction_ids': int(key_check.loc['IDs that repeat', 'transactions']),
    'each repeated ID appears exactly twice': yes_no((copies_per_id == 2).all()),
    'copies sit in adjacent rows': yes_no((row_gap == 1).all()),
    'same row positions in both tables': yes_no(same_positions and same_ids_at_positions),
    'every ID in one table exists in the other': yes_no(same_id_sets),
    'rows that are exact full-row copies': exact_copy_rows,
    'trade_date conflicts after normalising': int(conflicts_per_column['trade_date']),
    'event_timestamp conflicts after normalising': int(conflicts_per_column['event_timestamp']),
    'pairs with a real conflict': int(pair_has_real_conflict.sum()),
    'conflicts: settlement_date': int(conflicts_per_column['settlement_date']),
    'conflicts: price': int(conflicts_per_column['price']),
    'conflicts: quantity': int(conflicts_per_column['quantity']),
    'conflicts: currency': int(conflicts_per_column['currency']),
    'conflicts: settlement_note': int(conflicts_per_column['settlement_note']),
    'repeated rows: BOOKED': int(status_repeated.get('BOOKED', 0)),
    'repeated rows: CANCELLED': int(status_repeated.get('CANCELLED', 0)),
})
NOTEBOOK_RESULTS.update({f'pipeline pairs differ: {column}': int(pipe_differs_per_column[column])
                         for column in pipe_columns})
NOTEBOOK_RESULTS.update({
    'positions as_of_date values': basics['positions']['as_of_date values'],
    'comparison_positions as_of_date values': basics['comparison_positions']['as_of_date values'],
    'positions distinct account_ids': basics['positions']['distinct account_ids'],
    'comparison_positions distinct account_ids': basics['comparison_positions']['distinct account_ids'],
    'positions distinct security_ids': basics['positions']['distinct security_ids'],
    'comparison_positions distinct security_ids': basics['comparison_positions']['distinct security_ids'],
    'merge: in both': int(merge_counts['in both']),
    'merge: only in positions': int(merge_counts['only in positions']),
    'merge: only in comparison_positions': int(merge_counts['only in comparison_positions']),
    'book-only rows with zero quantity': int((book_only['quantity_number'] == 0).sum()),
    'book-only quantity min': book_only_summary['min'],
    'book-only quantity max': book_only_summary['max'],
    'book-only quantity median (about)': round(book_only_summary['50%'], 2),
    'book-only quantity mean (about)': round(book_only_summary['mean'], 2),
})
NOTEBOOK_ESTIMATES = {
    'pairs differing only by date format': int(pair_differs_only_by_format.sum()),
    'account_ids in positions missing from accounts': len(orphan_account_ids),
    'security_ids in positions missing from securities': len(orphan_security_ids),
}

cross_check_rows = []
for check, manual_value in MANUAL_RESULTS.items():
    notebook_value = NOTEBOOK_RESULTS.get(check, 'not computed')
    cross_check_rows.append({'check': check, 'our manual result': manual_value,
                             'this notebook': notebook_value,
                             'match?': yes_no(notebook_value == manual_value)})
for check, manual_value in MANUAL_ESTIMATES.items():
    cross_check_rows.append({'check': check, 'our manual result': manual_value,
                             'this notebook': NOTEBOOK_ESTIMATES[check],
                             'match?': 'estimate only'})

cross_check = pd.DataFrame(cross_check_rows)
cross_check.to_csv(OUTPUTS / '01_cross_check.csv', index=False)

print('match:', (cross_check['match?'] == 'yes').sum(),
      '| mismatch:', (cross_check['match?'] == 'no').sum(),
      '| estimate only:', (cross_check['match?'] == 'estimate only').sum())
cross_check

**Mismatches and estimates.** In this run there is **no mismatch**: all 51 scored checks agree with our manual results. For the three numbers we had only estimated: (1) pairs that differ only by date format are 28, as we guessed; (2) 195 account_ids in positions are missing from accounts, exactly our lower bound; (3) 15 security_ids are missing from securities, well above our lower bound of 3, because the guess only counted the difference in distinct IDs (6,015 minus 6,012) and ignored that the security master also contains securities nobody holds. If a later run shows a mismatch, rerun the cell that produces that number and print the rows behind it; a different date normalisation or a forgotten filter is the most likely cause on either side.

### 9.3 Live headline numbers

The summary box at the top was written from a run of this notebook. The cell below prints the same headline numbers live, so after any rerun you can check that the box is still true.

In [ ]:
print(f"Integrity: {(shape_check['result'] == 'PASS').sum()} of {len(shape_check)} tables match manifest.csv")
print(f"transaction_id: {NOTEBOOK_RESULTS['distinct transaction_ids']:,} distinct in "
      f"{len(transactions):,} rows; {NOTEBOOK_RESULTS['repeated transaction_ids']:,} IDs repeat")
print(f"Repeated pairs: {pair_types['exact copies']:,} exact copies, "
      f"{pair_types['differ only by date format']:,} differ only by date format, "
      f"{pair_types['real conflict (at least one column)']:,} real conflicts")
print(f"Positions vs custodian: {merge_counts['in both']:,} in both, "
      f"{merge_counts['only in positions']:,} only in positions, "
      f"{merge_counts['only in comparison_positions']:,} only in comparison_positions")
print(f"Orphans in positions: {len(orphan_account_ids):,} account_ids ({rows_with_orphan_account:,} rows), "
      f"{len(orphan_security_ids):,} security_ids ({rows_with_orphan_security:,} rows)")
print(f"pandas and DuckDB agree on all {len(pandas_vs_duckdb)} checks: "
      f"{(pandas_vs_duckdb['agree?'] == 'yes').all()}")
print(f"Cross-check: {(cross_check['match?'] == 'yes').sum()} match, "
      f"{(cross_check['match?'] == 'no').sum()} mismatch")